---
title : "Talk with WISKI - tools"
execute:
    error: false
    eval: false
    output: asis
format:
    html:
        warning: false
        error: false
        code-fold: false
        css: style.css
        panel: center
---

_Leave `eval: false, output: asis`for Github CI, remove to generate docs locally_

In de documentatie laten we voorbeelden zien hoe je verschillende tools gebruikt. 

In [1]:
# zorg dat alle variable goed staan
import json
from setup_quarto import setup_dot_env

ctx = setup_dot_env()

## 1. Parameters

Met de tool `get_parameters` kan je de parameters van WISKI opvragen.

In [2]:
from talk_with_wiski.models import GetLocationsInput, GetParametersInput
from talk_with_wiski.tools import get_locations, get_parameters

Dit kan voor een aantal stations

In [3]:
get_parameter_input = GetParametersInput(station_no=["ZUN0034_KKL_OW", "0300_TDB"])
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.model_dump()

Of voor alle stations

In [4]:
import numpy as np

get_parameter_input = GetParametersInput()
get_parameter_output = await get_parameters(ctx, get_parameter_input)

data_dict = get_parameter_output.data
parameters = np.unique(
    [row["stationparameter_longname"].split(" [")[0].lower() for row in data_dict]
)
print(parameters)

In [5]:
get_parameter_input = GetParametersInput(
    station_no="BRE0301_PMP1",
)
get_parameter_output = await get_parameters(ctx, get_parameter_input)

get_parameter_output.data

## 2. Locations

Met de tool `get_locations` kan je de parameters van WISKI opvragen.

In [6]:
get_locations_input = GetLocationsInput(max_number_of_station_ids=5)

ctx = setup_dot_env()

get_locations_output = await get_locations(ctx, get_locations_input)

get_locations_output.model_dump()

### Using geocoding is also possible

This uses `from geopy.geocoders import Nominatim`, <br>
which uses the OSM API, [See the policy](https://operations.osmfoundation.org/policies/nominatim/)<br>
 -> `No heavy uses (an absolute maximum of 1 request per second).` 

In [7]:
get_locations_input = GetLocationsInput(
    requested_geolocation=["Loon op Zand", 5.0],
    returnfields="station_no,station_id,station_name,stationparameter_name,station_latitude,station_longitude,site_name",
)
print(input)
get_locations_output = await get_locations(ctx, input)
for item in get_locations_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_locations_output.data)}")

## 3. Sites

Met de tool `get_sites` kun je de beschikbare sites van WISKI opvragen.

Hier een aantal voorbeelden.

In [8]:
from talk_with_wiski.models import GetSitesInput
from talk_with_wiski.tools import get_sites

get_sites_input = GetSitesInput(
    returnfields="site_no,site_id,site_name",
)
get_sites_output = await get_sites(ctx, get_sites_input)
for item in get_sites_output.data:
    print(json.dumps(item, indent=2))
print(f"Aantal records returned {len(get_sites_output.data)}")

## 4. Group lists

In [9]:
from talk_with_wiski.models import GetGroupListInput, GetGroupListOutput
from talk_with_wiski.tools import get_group_list

# Test end time before start time
get_group_list_input = GetGroupListInput(group_type="timeseries")

get_group_list_output: GetGroupListOutput = await get_group_list(
    ctx, get_group_list_input
)
get_group_list_output.model_dump()

## 5. ParametertypeList

In [10]:
from talk_with_wiski.models import GetParametertypeListInput, GetParametertypeListOutput
from talk_with_wiski.tools import get_parametertype_list

In [11]:
print("Test ParametertypeList: get ids for given names")
get_parametertype_list_input = GetParametertypeListInput(
    parametertype_name=["Evaporation", "WNS*"],
)
print(get_parametertype_list_input)
get_parametertype_list_output: GetParametertypeListOutput = (
    await get_parametertype_list(ctx, get_parametertype_list_input)
)

print(f"Aantal parametertype gevonden {len(get_parametertype_list_output.data)}")
get_parametertype_list_output.model_dump()

## 6. Timeseries list

Met de tool `get_timeseries_list` kan je de beschikbare timeseries ophalen voor een locatie, site en/of een parameter.

In [12]:
from talk_with_wiski.models import GetTimeseriesListInput, GetTimeseriesListOutput
from talk_with_wiski.tools import get_timeseries_list

Hiervoor gebruiken we de eerste op gehaalde parameter en locatie combinatie: 

In [13]:
previously_retrieved_station = get_parameter_output.data[0]["station_no"]
previously_retrieved_parameter = [
    data["parametertype_name"] for data in get_parameter_output.data
]
previously_retrieved_station, previously_retrieved_parameter

In [14]:
get_timeseries_list_input = GetTimeseriesListInput(
    station_no=previously_retrieved_station,
    parametertype_name=previously_retrieved_parameter,
)
get_timeseries_list_output: GetTimeseriesListOutput = await get_timeseries_list(
    ctx, get_timeseries_list_input
)

get_timeseries_list_output.model_dump()

In [15]:
get_timeseries_list_output.model_dump()["data"][0].keys()

## 7. Timeseries values

Met de tool `get_timeseries_values` kan je de timeseries data ophalen voor een timeseries_id.

In [16]:
from talk_with_wiski.models import GetTimeseriesValuesInput, GetTimeseriesValuesOutput
from talk_with_wiski.tools import get_timeseries_values

In [17]:
get_timeseries_value_input = GetTimeseriesValuesInput(
    ts_id=[item["ts_id"] for item in get_timeseries_list_output.data],
    from_date="2025-11-01",
    period="P1D",
    # to_date="2025-11-06"
)
get_timeseries_value_output: GetTimeseriesValuesOutput = await get_timeseries_values(
    ctx, get_timeseries_value_input
)

get_timeseries_value_output.model_dump()

In [18]:
[item["ts_id"] for item in get_timeseries_list_output.data]

## 8. Timeseries Type

Met de tool `get_timeseries_Type` kan je de informatie ophalen of het type tijdserie.

In [19]:
from talk_with_wiski.models import GetTimeseriesTypeInput, GetTimeseriesTypeOutput
from talk_with_wiski.tools import get_timeseries_type

We reuse the timeseries types from when we retrieved the timeseries list

In [20]:
ts_type_id = get_timeseries_list_output.data[0]["ts_type_id"]
ts_type_id

In [21]:
get_timeseries_type_input = GetTimeseriesTypeInput(ts_type_id=ts_type_id)
get_timeseries_type_output: GetTimeseriesTypeOutput = await get_timeseries_type(
    ctx, get_timeseries_type_input
)
get_timeseries_type_output.model_dump()

## 9 Quality Codes

De data heeft een quality code, die iets zegt over de betrouwbaarheid van de data.

In [22]:
# to do analysis on timeseries
from talk_with_wiski.models import (
    AnalyseTimeseriesQualityCodeInput,
    AnalyseTimeseriesQualityCodeOutput,
)
from talk_with_wiski.tools import (
    analyse_timeseries_quality_code,
)

We hergebruike het eerder gebruikte id 

In [23]:
ts_id = get_timeseries_list_output.data[0]["ts_id"]
ts_id

In [24]:
analyse_timeseries_input = AnalyseTimeseriesQualityCodeInput(
    ts_id=ts_id,  # belongs to stat_no "1198_WTH_BOV"
    from_date="2023-11-01",
    period="P1D",  # dit mag niet icm from_date.. to_date
    max_number_of_tsids_to_retrieve=20,
    max_number_of_values_per_tsid=1000000,
)
analyse_timeseries_output: AnalyseTimeseriesQualityCodeOutput = (
    await analyse_timeseries_quality_code(ctx, analyse_timeseries_input)
)
analyse_timeseries_output.model_dump()

## 10 Metadata 

Naast de standaard informatie die bij een tijdserie hoort, is er ook metadata beschikbaar. 
De meeste `get_..` tools waren gericht op het doel: tijdserie data ophalen. De metadata tool is gericht op alle informatie er omheen, deze kan gebruikt worden voor zowel tijdseries, locaties, parameters etc.

Ten eerste tijdseries:

In [25]:
from talk_with_wiski.models import GetMetadataInput, GetMetadataOutput
from talk_with_wiski.tools import get_metadata

In [26]:
get_metadata_input = GetMetadataInput(
    ts_id=ts_id,
)
get_metadata_output: GetMetadataOutput = await get_metadata(ctx, get_metadata_input)

get_metadata_output.model_dump()

Naast metadata voor timeseries, kunnen we ook metadata voor sites en stations ophalen.

In [27]:
# Extract station_no_list values from the previous metadata output
station_no_list = [item["station_no"] for item in get_metadata_output.data]
site_no_list = [item["site_no"] for item in get_metadata_output.data]
station_no_list

In [28]:
input = GetMetadataInput(
    station_no=station_no_list,
)
station_metadata_list_of_dicts: GetMetadataOutput = await get_metadata(ctx, input)
station_metadata_list_of_dicts.model_dump()

In [29]:
site_no_list

In [30]:
get_metadata_input = GetMetadataInput(
    site_no=site_no_list,
)
get_metadata_output: GetMetadataOutput = await get_metadata(ctx, get_metadata_input)
get_metadata_output.model_dump()

## 11 Analyse Timeseries Statistics

Naast alleen data ophalen kunnen we ook data analyseren, hiervoor gebruiken we een vergelijkbare definitie als het ophalen van tijdseries. Onder de motorkap roepen de analyse tools de get timeseries aan, vervolgens analyseren ze deze data en geven het terug.

In [31]:
from talk_with_wiski.models import (
    AnalyseTimeseriesStatsInput,
    AnalyseTimeseriesStatsOutput,
)
from talk_with_wiski.tools import analyse_timeseries_statistics

In [32]:
input_analyse_timeseries_statistics = AnalyseTimeseriesStatsInput(
    ts_id=ts_id,
    from_date="2023-11-01",
    period="P3M",
)

output_analyse_timeseries_statistics: AnalyseTimeseriesStatsOutput = (
    await analyse_timeseries_statistics(ctx, input_analyse_timeseries_statistics)
)

output_analyse_timeseries_statistics.model_dump()

## 12 Analyse Timeseries Threshold

In [33]:
from talk_with_wiski.models import (
    AnalyseTimeseriesThresholdsInput,
    AnalyseTimeseriesThresholdsOutput,
)
from talk_with_wiski.tools import analyse_timeseries_thresholds

In [34]:
analyse_timeseries_input = AnalyseTimeseriesThresholdsInput(
    ts_id=ts_id,
    from_date="2023-11-01",
    period="P3M",
    threshold=5.3,
)

output_analyse_timeseries_thresholds: AnalyseTimeseriesThresholdsOutput = (
    await analyse_timeseries_thresholds(ctx, analyse_timeseries_input)
)
output_analyse_timeseries_thresholds.model_dump()

## 13 Analyse Timeseries trend

In [35]:
from talk_with_wiski.models import (
    AnalyseTimeseriesTrendInput,
    AnalyseTimeseriesTrendOutput,
)
from talk_with_wiski.tools import analyse_timeseries_trend

In [36]:
# for statistics and trend, same object can be used
input_analyse_timeseries_trend = AnalyseTimeseriesTrendInput(
    ts_id=ts_id,
    from_date="2023-11-01",
    period="P3M",
)

output_analyse_timeseries_trend: AnalyseTimeseriesTrendOutput = (
    await analyse_timeseries_trend(ctx, input_analyse_timeseries_trend)
)

output_analyse_timeseries_trend.model_dump()

Meer informatie over hoe de analyse tools werken? zie ook [dit voorbeeld](analyse_timeseries_futher_examples.ipynb)

## 14 Analyse Timeseries availability
Naast het ophalen van de tijdseries data zelf, is het sneller om te checken wanneer data beschikbaar is. 

In [37]:
from talk_with_wiski.models import (
    AnalyseTimeseriesAvailabilityInput,
    AnalyseTimeseriesAvailabilityOutput,
)
from talk_with_wiski.tools import analyse_timeseries_availability

In [38]:
# for statistics and trend, same object can be used
input_analyse_timeseries_availability = AnalyseTimeseriesAvailabilityInput(
    ts_id=ts_id,
    from_date="2023-11-01",
    period="P3M",
)

output_analyse_timeseries_availability: AnalyseTimeseriesAvailabilityOutput = (
    await analyse_timeseries_availability(ctx, input_analyse_timeseries_availability)
)

output_analyse_timeseries_availability.model_dump()

Indien er gezocht wordt buiten de periode dat er data is, wordt het bereik van data terug gegeven

In [39]:
input_analyse_timeseries_availability = AnalyseTimeseriesAvailabilityInput(
    ts_id=ts_id,
    from_date="1990-11-01",
    period="P1M",
)

output_analyse_timeseries_availability: AnalyseTimeseriesAvailabilityOutput = (
    await analyse_timeseries_availability(ctx, input_analyse_timeseries_availability)
)

output_analyse_timeseries_availability.model_dump()